## 0. Read Data for Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

In [ ]:
import sqlite3

In [ ]:
con = sqlite3.connect('/Users/adigonitodoulou/Desktop/Data_Analysis_Projects/Amazon/database.sqlite')

In [ ]:
pd.read_sql_query(
"SELECT name FROM sqlite_master WHERE type='table';",
con
)

In [ ]:
df = pd.read_sql_query('SELECT * FROM REVIEWS', con)

In [ ]:
df.shape

In [ ]:
df.columns

## 1. Data Cleaning

In [ ]:
df[df['HelpfulnessNumerator'] > df['HelpfulnessDenominator']] #invalid row

In [ ]:
df_valid = df[df['HelpfulnessNumerator'] <= df['HelpfulnessDenominator']]

In [ ]:
df_valid.shape

In [ ]:
df_valid.duplicated(['UserId', 'ProfileName', 'Time', 'Text'])

In [ ]:
df_valid[df_valid.duplicated(['UserId', 'ProfileName', 'Time', 'Text'])]

In [ ]:
data = df_valid.drop_duplicates(subset= ['UserId', 'ProfileName', 'Time', 'Text']).copy()

In [ ]:
data.shape

In [ ]:
data.dtypes

In [ ]:
data['Time'] = pd.to_datetime(data['Time'], unit='s')

## 2. EDA

## 2.1 Who are the most active reviewers

In [ ]:
data.columns

In [ ]:
data['ProfileName'].nunique()

In [ ]:
data['UserId'].nunique()

In [ ]:
recommend_df = data.groupby(['UserId']).agg({
    'Summary' : 'count', 'Text' : 'count', 'Score' : 'mean', 'ProductId' : 'count'
}).sort_values(by = 'ProductId', ascending = False)

In [ ]:
recommend_df.columns = ['number_of_summaries', 'number_text', 'avg_score', 'number_of_product_reviews']

In [ ]:
recommend_df.index[0:10]

In [ ]:
recommend_df['number_of_product_reviews'].iloc[:10].values

In [ ]:
plt.bar(recommend_df.index[0:10], recommend_df['number_of_product_reviews'].iloc[:10].values)
plt.title('Top 10 Most Active Reviewers')
plt.xlabel('User ID')
plt.ylabel('Number of Reviews')
plt.xticks(rotation = 'vertical')

In [ ]:
data.columns

In [ ]:
len(data['ProductId'].unique())

In [ ]:
top_10_review_share = (
    recommend_df['number_of_product_reviews']
    .head(10)
    .sum()
    / len(data)
    * 100
)

In [ ]:
print(
f'The 10 most active reviewers contributed {top_10_review_share:.2f}% of all cleaned reviews.'
)

## 2.2 How are ratings distributed across frequently reviewed products

In [ ]:
prod_count = data['ProductId'].value_counts().to_frame().reset_index()

#### Frequently Reviewed Product Definition
Products with more than 500 review records are classified as frequently reviewed products.

In [ ]:
freq_prod_ids = prod_count[prod_count['count'] > 500]['ProductId'].values

In [ ]:
freq_prod_ids

In [ ]:
freq_prod_df = data[data['ProductId'].isin(freq_prod_ids)].copy()

In [ ]:
freq_prod_df.columns

In [ ]:
sns.countplot(y = 'ProductId', data = freq_prod_df, hue = 'Score')
plt.title('Review Score Distribution by Product')

In [ ]:
product_rating_summary = (
    freq_prod_df
    .groupby('ProductId')
    .agg(
        number_of_reviews=('Score', 'count'),
        average_rating=('Score', 'mean'),
        median_rating=('Score', 'median')
    )
    .sort_values('average_rating', ascending=False)
)

product_rating_summary

In [ ]:
highest_rated_products = product_rating_summary.head(5)
highest_rated_products

In [ ]:
lowest_rated_products = (
    product_rating_summary
    .sort_values('average_rating')
    .head(5)
)
lowest_rated_products

In [ ]:
highest_product = product_rating_summary['average_rating'].idxmax()
highest_rating = product_rating_summary['average_rating'].max()

lowest_product = product_rating_summary['average_rating'].idxmin()
lowest_rating = product_rating_summary['average_rating'].min()

print(
    f'Among products with more than 500 reviews, '
    f'{highest_product} has the highest average rating '
    f'of {highest_rating:.2f} stars.'
)

print(
    f'{lowest_product} has the lowest average rating '
    f'of {lowest_rating:.2f} stars.'
)

## 2.3 Do frequent and non-frequent reviewers exhibit different rating behaviour

In [ ]:
x = data['UserId'].value_counts()

#### Reviewer Segmentation
Reviewers with more than 50 reviews are classified as frequent reviewers. All remaining reviewers are classified as non-frequent reviewers.

In [ ]:
data['reviewer_type'] = data['UserId'].apply(lambda user : 'Frequent' if x[user]>50 else 'Non-Frequent')

In [ ]:
not_freq_df = data[data['reviewer_type'] == 'Non-Frequent'].copy()
freq_df = data[data['reviewer_type'] == 'Frequent'].copy()

In [ ]:
freq_pct = freq_df['Score'].value_counts(normalize=True).sort_index() * 100

In [ ]:
freq_pct.plot(
    kind='bar',
    title='Review Score Distribution: Frequent Reviewers'
)

plt.xlabel('Review Score')
plt.ylabel('Percentage of Reviews (%)')
plt.ylim(0, 100)

In [ ]:
not_freq_pct =  not_freq_df['Score'].value_counts(normalize=True).sort_index() * 100

In [ ]:
not_freq_pct.plot(
    kind='bar',
    title='Review Score Distribution: Non-Frequent Reviewers'
)

plt.xlabel('Review Score')
plt.ylabel('Percentage of Reviews (%)')
plt.ylim(0, 100)

In [ ]:
rating_summary = (
    data
    .groupby('reviewer_type')['Score']
    .agg(
        number_of_reviews='count',
        average_rating='mean',
        median_rating='median',
        rating_std='std'
    )
    .round(2)
)

rating_summary

In [ ]:
frequent_avg_rating = freq_df['Score'].mean()
non_frequent_avg_rating = not_freq_df['Score'].mean()

rating_difference = frequent_avg_rating - non_frequent_avg_rating

print(
    f'Frequent reviewers gave an average rating of '
    f'{frequent_avg_rating:.2f} stars.'
)

print(
    f'Non-frequent reviewers gave an average rating of '
    f'{non_frequent_avg_rating:.2f} stars.'
)

print(
    f'The difference in average ratings is '
    f'{abs(rating_difference):.2f} stars.'
)

In [ ]:
if rating_difference > 0:
    print(
        'Frequent reviewers give slightly higher ratings '
        'on average than non-frequent reviewers.'
    )
elif rating_difference < 0:
    print(
        'Frequent reviewers give slightly lower ratings '
        'on average than non-frequent reviewers.'
    )
else:
    print(
        'The two reviewer groups have the same average rating.'
    )

In [ ]:
frequent_five_star_pct = (
    freq_df['Score'].eq(5).mean() * 100
)

non_frequent_five_star_pct = (
    not_freq_df['Score'].eq(5).mean() * 100
)

print(
    f'Five-star reviews among frequent reviewers: '
    f'{frequent_five_star_pct:.2f}%'
)

print(
    f'Five-star reviews among non-frequent reviewers: '
    f'{non_frequent_five_star_pct:.2f}%'
)

In [ ]:
freq_pct = (
    freq_df['Score']
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

not_freq_pct = (
    not_freq_df['Score']
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

rating_comparison = pd.DataFrame({
    'Frequent': freq_pct,
    'Non-Frequent': not_freq_pct
})

rating_comparison.plot(
    kind='bar',
    figsize=(9, 5)
)

plt.title('Review Score Distribution by Reviewer Type')
plt.xlabel('Review Score')
plt.ylabel('Percentage of Reviews (%)')
plt.xticks(rotation=0)
plt.legend(title='Reviewer Type')
plt.show()

## 2.4 Do frequent reviewers write longer reviews

In [ ]:
data[['UserId', 'ProductId', 'Text']]

In [ ]:
def calculate_length(text):
    return len(text.split())

In [ ]:
data['Review_Word_Count'] = data['Text'].apply(calculate_length)

In [ ]:
not_freq_data = data[data['reviewer_type'] == 'Non-Frequent'].copy()
freq_data = data[data['reviewer_type'] == 'Frequent'].copy()

In [ ]:
fig = plt.figure()
fig.suptitle('Review Word Count: Frequent vs Non-Frequent Reviewers')

ax1 = fig.add_subplot(121)
ax1.boxplot(freq_data['Review_Word_Count'])
ax1.set_xlabel('Frequent Reviewers')
ax1.set_ylabel('Review Word Count')
ax1.set_ylim(0,600)

ax2 = fig.add_subplot(122)
ax2.boxplot(not_freq_data['Review_Word_Count'])
ax2.set_xlabel('Non-Frequent Reviewers')
ax2.set_ylim(0,600)




In [ ]:
word_count_summary = (
    data
    .groupby('reviewer_type')['Review_Word_Count']
    .agg(
        number_of_reviews='count',
        mean_word_count='mean',
        median_word_count='median',
        minimum='min',
        maximum='max'
    )
    .round(2)
)

word_count_summary

In [ ]:
frequent_median_words = (
    freq_data['Review_Word_Count'].median()
)

non_frequent_median_words = (
    not_freq_data['Review_Word_Count'].median()
)

frequent_mean_words = (
    freq_data['Review_Word_Count'].mean()
)

non_frequent_mean_words = (
    not_freq_data['Review_Word_Count'].mean()
)

print(
    f'Frequent reviewers write a median of '
    f'{frequent_median_words:.1f} words per review.'
)

print(
    f'Non-frequent reviewers write a median of '
    f'{non_frequent_median_words:.1f} words per review.'
)

print(
    f'Mean word count for frequent reviewers: '
    f'{frequent_mean_words:.1f}'
)

print(
    f'Mean word count for non-frequent reviewers: '
    f'{non_frequent_mean_words:.1f}'
)

In [ ]:
word_count_difference = (
    frequent_median_words - non_frequent_median_words
)

if word_count_difference > 0:
    print(
        f'Frequent reviewers write longer reviews, with a '
        f'median difference of {word_count_difference:.1f} words.'
    )
elif word_count_difference < 0:
    print(
        f'Non-frequent reviewers write longer reviews, with a '
        f'median difference of {abs(word_count_difference):.1f} words.'
    )
else:
    print(
        'Both groups have the same median review word count.'
    )

## 2.5 Sentiment Analysis

In [ ]:
from textblob import TextBlob

In [ ]:
data.shape

In [ ]:
sample = data.sample(n=50000, random_state=42).copy()

In [ ]:
sample.head(2)

In [ ]:
polarity = []

for text in sample['Text']:
    if pd.isna(text):
        polarity.append(np.nan)
    else:
        polarity.append(
            TextBlob(str(text)).sentiment.polarity
        )

In [ ]:
sample['polarity'] = polarity

In [ ]:
sample_negative = sample[sample['polarity'] < 0 ]
sample_positive = sample[sample['polarity'] > 0 ]

In [ ]:
from collections import Counter

In [ ]:
Counter(sample_negative['Text']).most_common(10)

In [ ]:
Counter(sample_positive['Text']).most_common(10)

In [ ]:
sentiment_counts = pd.Series({
    'Positive': (sample['polarity'] > 0).sum(),
    'Neutral': (sample['polarity'] == 0).sum(),
    'Negative': (sample['polarity'] < 0).sum()
})

sentiment_counts.plot(
    kind='bar',
    title='Sentiment Distribution in Review Texts'
)

plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=0)

In [ ]:
sentiment_pct = (
    sentiment_counts
    .div(sentiment_counts.sum())
    .mul(100)
    .round(2)
)

sentiment_pct

In [ ]:
for sentiment, percentage in sentiment_pct.items():
    print(f'{sentiment}: {percentage:.2f}%')

In [ ]:
dominant_sentiment = sentiment_counts.idxmax()
dominant_percentage = sentiment_pct.loc[dominant_sentiment]

print(
    f'The dominant sentiment is {dominant_sentiment.lower()}, '
    f'representing {dominant_percentage:.2f}% of summaries '
    f'with available sentiment scores.'
)

In [ ]:
sentiment_pct.plot(
    kind='bar',
    figsize=(7, 5),
    title='Sentiment Distribution in Review Texts'
)

plt.xlabel('Sentiment')
plt.ylabel('Percentage of Review Texts (%)')
plt.xticks(rotation=0)
plt.show()